In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import datetime
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

In [2]:
raw_labels = pd.read_csv('/Users/eric/repos/aud/data/day_labels.csv')

features = raw_labels.copy(deep=True).reset_index(drop=True)

# Create additional label columns for shifted days
features['lapse_d0'] = features['lapse']
day_shift = [1,2,3,4,5,6,7]
for delta in day_shift:
    col_name = 'lapse_d'+str(delta)
    features[col_name]=features.groupby('subid')['lapse'].shift(-delta)

In [3]:
# Pre-allocate new feature columns
# Latest EMA value
ema_nums = [2,3,4,5,6,7,8,9,10]
for i in ema_nums:
    added_str = 'latest_ema_'+str(i)
    features[added_str] = np.nan
# Short-run mean EMA value (average of up to 3 most recent previous EMAs)
for i in ema_nums:
    added_str = 'srm_ema_'+str(i)
    features[added_str] = np.nan
# Long-run mean EMA value (average of all EMAs prior to the current day)
for i in ema_nums:
    added_str = 'lrm_ema_'+str(i)
    features[added_str] = np.nan
# Recent lapse
features["recent_lapse_1"] = np.nan
features["recent_lapse_3"] = np.nan
features["recent_lapse_5"] = np.nan

# Additional labels for prediction
# Lapse-within-window
features['lapse_w0'] = features['lapse']
features['lapse_w1'] = np.nan
features['lapse_w3'] = np.nan
features['lapse_w7'] = np.nan

# Loop through the dataset and add feature values
#for idx,row in features.iloc[1:5].iterrows():
for idx,row in features.iterrows():
    # General information
    subid = row.subid
    day = row.day
    current_df = features.query("subid==@subid & day<=@day").sort_values(by='day',ascending=False).copy(deep=True)
    current_df_nonan = current_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10'])
    past_df = features.query("subid==@subid & day < @day").sort_values(by='day',ascending=False).copy(deep=True)
    past_df_nonan = past_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10'])

    # Latest EMA values
    # Empty data case
    if current_df_nonan.shape[0]==0:
        # No need to do anything to set this row to NaN since it is preallocated as NaN
        pass
    # Non-empty data case
    else:
        recent_vals=current_df_nonan.iloc[0]
        # Put these values into the dataframe in the correct row
        raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].tolist()
        features.loc[idx,['latest_ema_2','latest_ema_3','latest_ema_4','latest_ema_5','latest_ema_6','latest_ema_7','latest_ema_8','latest_ema_9','latest_ema_10']]=raw_vals

    # Short-run mean EMA value (average of the last 3 non-NaN responses, does not include today)
    # Empty data case
    if current_df_nonan.shape[0]==0:
        # No need to do anything to set this row to NaN since it is preallocated as NaN
        pass
    # Non-empty data case
    else:
        # Take up to 3 datapoints (as available)
        end_idx = min(current_df_nonan.shape[0],3)
        #recent_vals=past_df_nonan.iloc[0:end_idx]
        recent_vals=current_df_nonan.iloc[0:end_idx]
        # Put these values into the dataframe in the correct row
        raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
        features.loc[idx,['srm_ema_2','srm_ema_3','srm_ema_4','srm_ema_5','srm_ema_6','srm_ema_7','srm_ema_8','srm_ema_9','srm_ema_10']]=raw_vals

    # Long-run mean EMA value (average of all non-NaN responses, does not include today)
    # Empty data case
    if current_df_nonan.shape[0]==0:
        # No need to do anything to set this row to NaN since it is preallocated as NaN
        pass
    # Non-empty data case
    else:
        #recent_vals=past_df_nonan
        recent_vals=current_df_nonan
        # Put these values into the dataframe in the correct row
        raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
        features.loc[idx,['lrm_ema_2','lrm_ema_3','lrm_ema_4','lrm_ema_5','lrm_ema_6','lrm_ema_7','lrm_ema_8','lrm_ema_9','lrm_ema_10']]=raw_vals

    # Day of week dummies
    # Leaving this in for now due to odd behavior in tree-based models
    #features.drop(labels='day_of_week_num_0',axis=1,inplace=True)

    # Recent lapse (lapse occurring within the last 1,3,5 days)
    end_idx = min(1,past_df.shape[0])
    if end_idx >0:
        recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
        recent_lapse = np.any(recent_vals)
    else:
        recent_lapse = False
    features.loc[idx,'recent_lapse_1']=recent_lapse
    
    end_idx = min(3,past_df.shape[0])
    if end_idx >0:
        recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
        recent_lapse = np.any(recent_vals)
    else:
        recent_lapse = False
    features.loc[idx,'recent_lapse_3']=recent_lapse

    end_idx = min(5,past_df.shape[0])
    if end_idx >0:
        recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
        recent_lapse = np.any(recent_vals)
    else:
        recent_lapse = False
    features.loc[idx,'recent_lapse_5']=recent_lapse

    # Labels
    # Create shifted label columns
    w1_vec = [row.lapse_d0,row.lapse_d1]
    if any(np.isnan(w1_vec)):
        val = np.nan
    else:
        val = int(np.any(w1_vec))
    features.loc[idx,'lapse_w1'] = val

    w3_vec = [row.lapse_d0,row.lapse_d1,row.lapse_d2,row.lapse_d3]
    if any(np.isnan(w3_vec)):
        val = np.nan
    else:
        val = int(np.any(w3_vec))
    features.loc[idx,'lapse_w3'] = val

    w7_vec = [row.lapse_d0,row.lapse_d1,row.lapse_d2,row.lapse_d3,row.lapse_d4,row.lapse_d5,row.lapse_d6,row.lapse_d7]
    if any(np.isnan(w7_vec)):
        val = np.nan
    else:
        val = int(np.any(w7_vec))
    features.loc[idx,'lapse_w7'] = val

# Drop columns
features.drop(labels=['lapse_before_mema','lapse_after_mema','mema_arrival','lapse_start','lapse_end'],axis=1,inplace=True)
# Take a look
features.head()

,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7
0,1,0,0.333333,1.000000,1.000000,0.666667,0.5,0.7,0.4,0.4,0.3,0,0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,1.000000,1.000000,0.666667,0.5,0.7,0.4,0.4,0.3,0.333333,1.000000,1.000000,0.666667,0.500000,0.700000,0.400000,0.4,0.3,0.333333,1.000000,1.000000,0.666667,0.50,0.700000,0.400000,0.400,0.3,False,False,False,0,0.0,0.0,0.0
1,1,1,0.166667,0.333333,0.333333,0.666667,0.5,0.2,0.4,0.6,0.3,0,0,0,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.333333,0.333333,0.666667,0.5,0.2,0.4,0.6,0.3,0.250000,0.666667,0.666667,0.666667,0.500000,0.450000,0.400000,0.5,0.3,0.250000,0.666667,0.666667,0.666667,0.50,0.450000,0.400000,0.500,0.3,False,False,False,0,0.0,0.0,0.0
2,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.333333,0.333333,0.666667,0.5,0.2,0.4,0.6,0.3,0.250000,0.666667,0.666667,0.666667,0.500000,0.450000,0.400000,0.5,0.3,0.250000,0.666667,0.666667,0.666667,0.50,0.450000,0.400000,0.500,0.3,False,False,False,0,0.0,0.0,0.0
3,1,3,0.166667,0.000000,0.000000,0.416667,0.5,0.2,0.3,0.5,0.3,0,0,0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.000000,0.000000,0.416667,0.5,0.2,0.3,0.5,0.3,0.222222,0.444444,0.444444,0.583333,0.500000,0.366667,0.366667,0.5,0.3,0.222222,0.444444,0.444444,0.583333,0.50,0.366667,0.366667,0.500,0.3,False,False,False,0,0.0,0.0,0.0
4,1,4,0.166667,0.083333,0.333333,0.666667,0.7,0.7,0.6,0.4,0.3,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.083333,0.333333,0.666667,0.7,0.7,0.6,0.4,0.3,0.166667,0.138889,0.222222,0.583333,0.566667,0.366667,0.433333,0.5,0.3,0.208333,0.354167,0.416667,0.604167,0.55,0.450000,0.425000,0.475,0.3,False,False,False,0,0.0,0.0,0.0


In [4]:
display(features[['subid','day','ema_2','recent_lapse_1','recent_lapse_3','recent_lapse_5','lapse_d0','lapse_d1','lapse_d2','lapse_d3','lapse_d4','lapse_d5','lapse_d6','lapse_d7','lapse_w0','lapse_w1','lapse_w3','lapse_w7']].iloc[86:110])

,subid,day,ema_2,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,lapse_w0,lapse_w1,lapse_w3,lapse_w7
86,1,86,NaN,False,False,False,0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0,0.0,0.0,NaN
87,1,87,0.083333,False,False,False,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN
88,1,88,0.166667,False,False,False,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN
89,1,89,0.166667,False,False,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
90,2,0,0.000000,False,False,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
91,2,1,0.250000,False,False,False,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0,0.0,1.0
92,2,2,0.333333,False,False,False,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0.0,0.0,1.0
93,2,3,0.083333,False,False,False,0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0,0.0,0.0,1.0
94,2,4,0.000000,False,False,False,0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0.0,0.0,1.0
95,2,5,0.000000,False,False,False,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,0.0,1.0,1.0


In [5]:
# Filter out the subjects with fewer than 10 EMAs
subject_info = pd.read_csv('/Users/eric/repos/aud/data/subject_info.csv')


included_subjects = subject_info.query("responsiveness >= 0.25").subid.unique()
#included_subjects = subject_info.query("mema_count >= 60").subid.to_numpy()
export_df = features.query("subid in @included_subjects").copy(deep=True)

# Low EMA drops
print(features.shape)
print(export_df.shape)

# Drop lines after the last EMA day or before the first
for idx,row in subject_info.iterrows():
    subid = row.subid
    mema_count = row.mema_count
    last_ema = row.last_morning_ema_day
    first_ema = row.first_morning_ema_day
    drop_sub_df = export_df.query("subid == @subid & (day > @last_ema | day < @first_ema)")
    idxs = drop_sub_df.index.to_numpy()
    export_df.drop(labels=idxs,axis=0,inplace=True)

print(export_df.shape)

# Drop NA entries in the EMA? Confirm that this is only a few cases
idxs1 = export_df.loc[export_df.latest_ema_2.isnull()].index.to_numpy()
idxs2 = export_df.loc[export_df.srm_ema_2.isnull()].index.to_numpy()
idxs3 = export_df.loc[export_df.recent_lapse_1.isnull()].index.to_numpy()
print(idxs1,idxs2,idxs3)
#print(idxs)
#print(len(idxs))

export_df.drop(labels=np.concatenate((idxs1,idxs2,idxs3)),axis=0,inplace=True)
print(export_df.shape)

# Column dropping
export_df.drop(labels = ['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10','lapse',
                         'lapse_d0', 'lapse_d1', 'lapse_d2', 'lapse_d3', 'lapse_d4', 'lapse_d5', 'lapse_d6', 'lapse_d7'],axis=1,inplace=True)
export_df.to_csv('/Users/eric/repos/aud/data/ml_labels.csv',index=False)

(13590, 61)
(13320, 61)
(12572, 61)
[] [] []
(12572, 61)


In [12]:
features.loc[88:91]

,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7
88,1,88,0.166667,0.166667,0.333333,0.250000,0.7,0.3,0.6,0.6,0.4,0,1,0,0,0,0,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.166667,0.166667,0.333333,0.250000,0.7,0.3,0.6,0.6,0.4,0.166667,0.305556,0.333333,0.388889,0.5,0.333333,0.60,0.60,0.4,0.198413,0.267196,0.284392,0.287037,0.503175,0.379365,0.544444,0.563492,0.392063,False,False,False,0,0.0,NaN,NaN
89,1,89,0.166667,0.333333,0.333333,0.083333,0.4,0.4,0.6,0.6,0.4,0,0,1,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.166667,0.333333,0.333333,0.083333,0.4,0.4,0.6,0.6,0.4,0.138889,0.277778,0.333333,0.305556,0.5,0.333333,0.60,0.60,0.4,0.197917,0.268229,0.285156,0.283854,0.501562,0.379688,0.545313,0.564063,0.392188,False,False,False,0,NaN,NaN,NaN
90,2,0,0.000000,0.000000,0.333333,0.333333,0.7,0.7,0.6,0.6,0.1,0,0,0,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.333333,0.333333,0.7,0.7,0.6,0.6,0.1,0.000000,0.000000,0.333333,0.333333,0.7,0.700000,0.60,0.60,0.1,0.000000,0.000000,0.333333,0.333333,0.700000,0.700000,0.600000,0.600000,0.100000,NaN,NaN,NaN,0,0.0,0.0,0.0
91,2,1,0.250000,0.083333,0.500000,0.333333,0.3,0.7,0.7,0.7,0.3,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.250000,0.083333,0.500000,0.333333,0.3,0.7,0.7,0.7,0.3,0.125000,0.041667,0.416667,0.333333,0.5,0.700000,0.65,0.65,0.2,0.125000,0.041667,0.416667,0.333333,0.500000,0.700000,0.650000,0.650000,0.200000,False,False,False,0,0.0,0.0,1.0


In [6]:
# Example for showing people who had missing responses at the beginning of their trajectories
test = export_df.loc[export_df.latest_ema_2.isnull()]
display(test)

,subid,day,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse,lapse_w0,lapse_w1,lapse_w3,lapse_w7
